# Quantized Retrieval-Augmented Medical Vision-Language Diagnosis

This notebook provides the setup and execution steps to run the pipeline on **Kaggle** utilizing a **GPU accelerator (Tesla T4 or P100)**.

## Step 1: Clone Repository / Prepare Workspace

If you have pushed this source code to GitHub, clone it directly into Kaggle's writable `/kaggle/working/` directory. Alternatively, if you uploaded the source folder as a Kaggle Dataset, copy the files.

In [ ]:
# Option A: Clone from GitHub (Recommended)
# !git clone https://github.com/YOUR_GITHUB_USERNAME/medical_vlm_pipeline.git
# %cd medical_vlm_pipeline/src

# Option B: If uploaded as a Kaggle Dataset
# !cp -r /kaggle/input/medical-vlm-pipeline/* /kaggle/working/
# %cd /kaggle/working/src

## Step 2: Install Medical Deep Learning Dependencies

Kaggle's environment already pre-installs massive libraries like `torch`, `torchvision`, `transformers`, `numpy`, `pandas`, `scikit-learn`, and `matplotlib`. We only need to install specific medical imaging and retrieval libraries, which will install **extremely fast** (under 1-2 minutes)!

In [ ]:
!pip install monai pydicom nibabel SimpleITK faiss-cpu qdrant-client evaluate rouge-score bert-score grad-cam

## Step 3: Run End-to-End Validation Pipeline Demo

Execute the validation demo containing product quantization indexing, FAISS retrieval, clinical report synthesis, and Grad-CAM explainability hooks.

In [ ]:
!python demo_pipeline.py

## Step 4: Import and Run Modules in Cell

You can import modules directly inside your Kaggle Notebook to run customized experiments.

In [ ]:
import sys
import torch
from medical_vlm_pipeline import PipelineConfig, MedicalVLMPipeline

# Check if GPU accelerator is active
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

config = PipelineConfig()
config.encoders.projection_dim = 128
config.retrieval.top_k = 2
config.retrieval.quantized = True

pipeline = MedicalVLMPipeline(config, num_classes=3, class_names=["Healthy", "Meningioma", "Glioma"])
pipeline = pipeline.to(device)
print("Pipeline initialized successfully on Kaggle GPU!")

## Step 5: Loading Real-world IU Chest X-rays Dataset

Use the dedicated `load_iu_chest_xray_cases` helper to load your actual IU Chest X-ray images (resized to 256, 299, 320, or 512) and their paired radiology reports from `cleaned_dataset.csv` in Kaggle.

In [ ]:
# Define paths to your uploaded Kaggle Dataset
import os

# Adjust the dataset name matching your Kaggle Input path
dataset_dir = "/kaggle/input/iu-chest-x-rays-cleaned"
csv_path = os.path.join(dataset_dir, "cleaned_dataset.csv")
images_dir = os.path.join(dataset_dir, "resized_images/256") # Choose 256, 299, 320, or 512

from medical_vlm_pipeline import load_iu_chest_xray_cases, MedicalCaseDataset

if os.path.exists(csv_path):
    # 1. Parse CSV and load medical cases
    cases = load_iu_chest_xray_cases(csv_path, images_dir)
    
    # 2. Wrap into PyTorch Dataset
    dataset = MedicalCaseDataset(cases)
    print(f"[SUCCESS] Loaded {len(dataset)} paired Chest X-ray images and clinical reports!")
    
    # 3. Retrieve and inspect the first case
    sample = dataset[0]
    print(f"Case ID:      {sample['case_id']}")
    print(f"Image Shape:  {sample['image'].shape}")
    print(f"Clinical Rep: {sample['report_text'][:120]}...")
    print(f"Projection:   {sample['label']}")
else:
    print(f"[WARNING] Dataset not found at: {csv_path}")
    print("Please add the 'IU Chest X-rays (Cleaned)' Dataset to your Kaggle Notebook input sidebar.")